# LANL 02 SINDy Baseline

Fit a baseline SINDy model on a selected LANL proxy-state cycle and save discovered equations into the LANL namespace.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import LANL_CONFIG
from src.derivatives import estimate_derivatives_df
from src.preprocess.common import standardize_columns
from src.sindy import build_polynomial_library, SINDyModel, relative_error, compute_rollout_metrics, rollout_polynomial
from src.utils.plotting import plot_signal_panel

RESULTS_DIR = LANL_CONFIG.results_dir
CYCLE_PATH = RESULTS_DIR / "lanl_selected_cycle.csv"
METADATA_PATH = RESULTS_DIR / "lanl_selected_cycle_metadata.json"
ROLLOUT_PLOT_PATH = RESULTS_DIR / "baseline_rollout.png"

POLY_DEGREE = 3
THRESHOLD = 1e-6
MAX_ITER = 15
DERIVATIVE_WINDOW = 101
DERIVATIVE_POLYORDER = 3
ROLLOUT_POINTS = 500


In [2]:
if not CYCLE_PATH.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError("Run notebooks/lanl/01_preprocess_and_segment.ipynb first.")

cycle_df = pd.read_csv(CYCLE_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
state_cols = metadata["state_columns"]
state_df = cycle_df[["time", *state_cols]].copy()
standardized_df, scaling = standardize_columns(state_df, state_cols)
derivatives, _ = estimate_derivatives_df(
    standardized_df,
    time_col="time",
    var_cols=state_cols,
    method="savgol",
    window=DERIVATIVE_WINDOW,
    polyorder=DERIVATIVE_POLYORDER,
    add_to_df=False,
)

X = standardized_df[state_cols].to_numpy(dtype=float)
Xdot = np.column_stack([derivatives[column] for column in state_cols])
library, descriptions = build_polynomial_library(X, degree=POLY_DEGREE, var_names=state_cols)
model = SINDyModel(threshold=THRESHOLD, max_iter=MAX_ITER)
diagnostics = model.fit(library, Xdot, descriptions)
Xdot_pred = model.predict(library)
equations = model.equations(state_cols)

rollout_len = min(ROLLOUT_POINTS, len(standardized_df))
rollout_time = standardized_df["time"].to_numpy(dtype=float)[:rollout_len]
rollout_true = X[:rollout_len]
rollout_pred = rollout_polynomial(model.coefficients, descriptions, X[0], rollout_time, {"state_1": state_cols[0], "state_2": state_cols[1]})
rollout_metrics = compute_rollout_metrics(rollout_true, rollout_pred)

summary = {
    "dataset": "lanl",
    "state_columns": state_cols,
    "state_labels": metadata["state_labels"],
    "scaling": scaling,
    "poly_degree": POLY_DEGREE,
    "threshold": THRESHOLD,
    "max_iter": MAX_ITER,
    "library_terms": descriptions,
    "equations": equations,
    "relative_error": [float(relative_error(Xdot[:, i], Xdot_pred[:, i])) for i in range(Xdot.shape[1])],
    "rmse_like": diagnostics["residuals"].tolist(),
    "rollout_metrics": rollout_metrics,
    "rollout_plot": str(ROLLOUT_PLOT_PATH),
}
(RESULTS_DIR / "baseline_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(RESULTS_DIR / "discovered_equations.txt").write_text("\n".join(equations) + "\n", encoding="utf-8")

print(json.dumps(summary, indent=2))


{
  "dataset": "lanl",
  "state_columns": [
    "tau_proxy",
    "V_proxy"
  ],
  "state_labels": [
    "tau_proxy",
    "V_proxy"
  ],
  "scaling": {
    "means": {
      "tau_proxy": 4.5788542077649215,
      "V_proxy": 1.2145123075213276e-05
    },
    "stds": {
      "tau_proxy": 2.2344452023232413,
      "V_proxy": 0.07878114371164875
    }
  },
  "poly_degree": 3,
  "threshold": 1e-06,
  "max_iter": 15,
  "library_terms": [
    "1",
    "tau_proxy",
    "V_proxy",
    "tau_proxy^2",
    "tau_proxy*V_proxy",
    "V_proxy^2",
    "tau_proxy^3",
    "tau_proxy^2*V_proxy",
    "tau_proxy*V_proxy^2",
    "V_proxy^3"
  ],
  "equations": [
    "dtau_proxy/dt = 5.435e-06*1 + 3.526e-02*V_proxy",
    "dV_proxy/dt = - 6.086e-04*1 - 3.308e-02*tau_proxy + 1.918e-03*V_proxy + 4.380e-05*tau_proxy^2 + 2.192e-04*tau_proxy*V_proxy + 1.256e-03*V_proxy^2 - 7.995e-06*tau_proxy^3 - 6.851e-06*tau_proxy^2*V_proxy - 2.868e-05*tau_proxy*V_proxy^2"
  ],
  "relative_error": [
    7.833553557110997e-16,
    

In [3]:
rollout_df = pd.DataFrame(
    {
        "time": rollout_time,
        f"true_{state_cols[0]}": rollout_true[:, 0],
        f"pred_{state_cols[0]}": rollout_pred[:, 0],
        f"true_{state_cols[1]}": rollout_true[:, 1],
        f"pred_{state_cols[1]}": rollout_pred[:, 1],
    }
)
fig, _ = plot_signal_panel(
    rollout_df,
    "time",
    [f"true_{state_cols[0]}", f"pred_{state_cols[0]}", f"true_{state_cols[1]}", f"pred_{state_cols[1]}"],
    "LANL baseline rollout",
    figsize=(12, 10),
)
fig.savefig(ROLLOUT_PLOT_PATH, dpi=200, bbox_inches="tight")
